# T05 - Motor Trend Car Road Tests

|                |   |
:----------------|---|
| **Nombre**     | Juan Pedro Ley Valdez  |
| **Fecha**      |  19 de febrero de 2026 |
| **Expediente** |  746385 |


**Instrucciones**    

*1.1 Realiza una regresión tomando 'mpg' como salida y eliminando la columna 'model'. Considera todos los demás factores como numéricos/ordinales.*


*   Calcula el R2 e interpreta los signos de los betas.
*   Realiza un train-test-split donde se use el 40% de los datos para entrenar. Calcula el R2 de entrenamiento y de prueba.
* Añade regularización L2 con un hiperparámetro lambda decidido por ti. Cambia este valor y compara con varios distintos los R2 de entrenamiento y de prueba.

*1.2 Repite el ejercicio anterior usando 'qsec' como salida.*  

*2.1 Realiza una regresión tomando 'mpg' como salida y eliminando la columna 'model'. Crea columnas dummies para los factores 'cyl', 'gear' y 'carb'.*



*   Calcula el R2 e interpreta los signos de los betas.

*   Realiza un train-test-split donde se use el 40% de los datos para entrenar. Calcula el R2 de entrenamiento y de prueba.

*2.2 Repite el ejercicio anterior usando 'qsec' como salida.*

*3.1 Compara los R2 de los ejercicios 1.1 & 2.1.*

*3.2 Compara los R2 de los ejercicios 1.2 & 2.2.*



## EDA

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
df = pd.read_excel('/content/motor_dataset.xlsx')

In [9]:
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


In [11]:
df = df.drop('model', axis=1)
X = df.drop('mpg', axis=1).values
y = df['mpg'].values

## 1.1 Regresión Lineal Múltiple e Interpretación de Betas
En esta sección calculamos el $R^2$ con el dataset completo y extraemos los coeficientes ($\beta$) para interpretar su impacto sobre la variable de salida `mpg`.

In [12]:
# Extraemos los nombres de las columnas de X para la interpretación:
columnas_X = df.drop('mpg', axis=1).columns

# Ajustar el modelo con todos los datos
modelo_completo = LinearRegression()
modelo_completo.fit(X, y)

# 1. Calcular R2
r2_completo = modelo_completo.score(X, y)
print(f"R2 del modelo completo: {r2_completo:.4f}\n")

# Extraer y mostrar los Betas
betas = pd.DataFrame({'Variable': columnas_X, 'Beta': modelo_completo.coef_})
print("Coeficientes (Betas):")
print(betas)

R2 del modelo completo: 0.8690

Coeficientes (Betas):
  Variable      Beta
0      cyl -0.111440
1     disp  0.013335
2       hp -0.021482
3     drat  0.787111
4       wt -3.715304
5     qsec  0.821041
6       vs  0.317763
7       am  2.520227
8     gear  0.655413
9     carb -0.199419


### Interpretación de los signos de los Betas
* **Signo Positivo (+):** Indica una relación directa. Si la variable independiente aumenta, el rendimiento (`mpg`) también tiende a aumentar (ej. `drat`, `qsec`, `vs`, `am`, `gear`).
* **Signo Negativo (-):** Indica una relación inversa. Si la variable aumenta, el rendimiento (`mpg`) disminuye. Esto tiene sentido físico en variables como el peso (`wt`), los cilindros (`cyl`) o los caballos de fuerza (`hp`), ya que un auto más pesado o más potente consume más combustible, reduciendo las millas por galón.

##  Train-Test Split (40% Entrenamiento)
Separamos los datos para evaluar la capacidad de generalización del modelo. Usamos el parámetro `train_size=0.4` como se solicitó.

In [13]:
# 2. Realizar el split (40% train, 60% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.4, random_state=42)

# Entrenar nuevo modelo solo con el set de entrenamiento
modelo_split = LinearRegression()
modelo_split.fit(X_train, y_train)

# Calcular R2 de entrenamiento y prueba
r2_train = modelo_split.score(X_train, y_train)
r2_test = modelo_split.score(X_test, y_test)

print(f"R2 de Entrenamiento: {r2_train:.4f}")
print(f"R2 de Prueba: {r2_test:.4f}")

R2 de Entrenamiento: 0.9982
R2 de Prueba: -7.1071


##  Regularización L2 (Ridge)
Añadimos penalización L2 a la regresión. Evaluaremos distintos valores para observar su efecto en el sobreajuste, causado seguramente por entrenar solo sobre el 40% de un dataset de 32 filas.

In [14]:
# 3. Probar distintos valores de lambda (alpha)
valores_lambda = [0.01, 0.1, 1.0, 10.0, 100.0]

print(f"{'Lambda':<10} | {'R2 Entrenamiento':<20} | {'R2 Prueba':<15}")
print("-" * 55)

for l in valores_lambda:
    modelo_ridge = Ridge(alpha=l)
    modelo_ridge.fit(X_train, y_train)

    r2_train_ridge = modelo_ridge.score(X_train, y_train)
    r2_test_ridge = modelo_ridge.score(X_test, y_test)

    print(f"{l:<10} | {r2_train_ridge:<20.4f} | {r2_test_ridge:<15.4f}")

Lambda     | R2 Entrenamiento     | R2 Prueba      
-------------------------------------------------------
0.01       | 0.9951               | -2.3331        
0.1        | 0.9794               | 0.2607         
1.0        | 0.9279               | 0.6311         
10.0       | 0.8634               | 0.6567         
100.0      | 0.8073               | 0.5990         


## 1.2 Regresión tomando 'qsec' como salida
A continuación, replicamos el análisis completo (Regresión Lineal, partición de datos al 40% y Regularización L2), pero esta vez utilizando `qsec` como nuestra variable dependiente ($y$) y el resto de las variables, incluyendo `mpg`, como variables independientes ($X$).

In [15]:
# --- Preparación de los nuevos datos ---

X_qsec = df.drop('qsec', axis=1).values
y_qsec = df['qsec'].values
columnas_X_qsec = df.drop('qsec', axis=1).columns

print("=== 1. MODELO COMPLETO E INTERPRETACIÓN DE BETAS ===")
modelo_qsec = LinearRegression()
modelo_qsec.fit(X_qsec, y_qsec)

print(f"R2 del modelo completo (qsec): {modelo_qsec.score(X_qsec, y_qsec):.4f}\n")

betas_qsec = pd.DataFrame({'Variable': columnas_X_qsec, 'Beta': modelo_qsec.coef_})
print("Coeficientes (Betas):")
print(betas_qsec.to_string(index=False))
print("\n" + "="*50 + "\n")

print("=== 2. TRAIN-TEST SPLIT (40% Entrenamiento) ===")
X_train_q, X_test_q, y_train_q, y_test_q = train_test_split(X_qsec, y_qsec, train_size=0.4, random_state=42)

modelo_split_qsec = LinearRegression()
modelo_split_qsec.fit(X_train_q, y_train_q)

print(f"R2 Entrenamiento: {modelo_split_qsec.score(X_train_q, y_train_q):.4f}")
print(f"R2 Prueba:        {modelo_split_qsec.score(X_test_q, y_test_q):.4f}")
print("\n" + "="*50 + "\n")

print("=== 3. REGULARIZACIÓN L2 (RIDGE) ===")
print(f"{'Lambda':<10} | {'R2 Entren.':<15} | {'R2 Prueba':<15}")
print("-" * 45)

valores_lambda = [0.01, 0.1, 1.0, 10.0, 100.0]
for l in valores_lambda:
    modelo_ridge_q = Ridge(alpha=l)
    modelo_ridge_q.fit(X_train_q, y_train_q)

    r2_tr_q = modelo_ridge_q.score(X_train_q, y_train_q)
    r2_te_q = modelo_ridge_q.score(X_test_q, y_test_q)

    print(f"{l:<10} | {r2_tr_q:<15.4f} | {r2_te_q:<15.4f}")

=== 1. MODELO COMPLETO E INTERPRETACIÓN DE BETAS ===
R2 del modelo completo (qsec): 0.8747

Coeficientes (Betas):
Variable      Beta
     mpg  0.069048
     cyl -0.362678
    disp -0.007501
      hp -0.001563
    drat -0.131064
      wt  1.496332
      vs  0.970035
      am -0.901186
    gear -0.201285
    carb -0.273598


=== 2. TRAIN-TEST SPLIT (40% Entrenamiento) ===
R2 Entrenamiento: 0.9989
R2 Prueba:        -1.0013


=== 3. REGULARIZACIÓN L2 (RIDGE) ===
Lambda     | R2 Entren.      | R2 Prueba      
---------------------------------------------
0.01       | 0.9975          | -0.1168        
0.1        | 0.9856          | 0.6878         
1.0        | 0.9347          | 0.7141         
10.0       | 0.8454          | 0.5044         
100.0      | 0.7837          | 0.4232         


### Interpretación de los signos de los Betas
La variable dependiente `qsec` representa el tiempo en segundos; por lo tanto, una reducción en su valor indica una mayor aceleración (un auto más rápido).

* **Coeficientes Negativos (Mejoran la aceleración):** Variables como los cilindros (`cyl`), los caballos de fuerza (`hp`) y el número de marchas (`gear`) tienen un signo negativo. Esto tiene sentido mecánico empírico: a mayor potencia o mayor cantidad de cilindros, el tiempo necesario para recorrer el cuarto de milla disminuye.
* **Coeficientes Positivos (Empeoran la aceleración):** El peso (`wt`) tiene un impacto positivo alto (1.496). Esto indica que, manteniendo todo lo demás constante, por cada unidad adicional de peso, el auto tarda aproximadamente 1.5 segundos más en completar el cuarto de milla. Un auto más pesado es más lento para acelerar.

## 2.1 Regresión con variables Dummy (Salida: `mpg`)
Convertimos los factores `cyl`, `gear` y `carb` en variables dummy (categóricas). Utilizamos `drop_first=True` para crear solo n-1 columnas nuevas, lo que nos permitirá interpretar los coeficientes de forma correcta en relación con una categoría base.

In [16]:
# === 2.1 REGRESIÓN CON DUMMIES (Salida: mpg) ===

# 1. Crear dummies. Asumimos que 'df' sigue intacto sin la columna 'model'
df_dummies = pd.get_dummies(df, columns=['cyl', 'gear', 'carb'], drop_first=True)

# Separar variables
X_d = df_dummies.drop('mpg', axis=1)
y_d = df_dummies['mpg']

# Ajustar modelo completo
modelo_d_mpg = LinearRegression()
modelo_d_mpg.fit(X_d.values, y_d.values)

print("=== MODELO CON DUMMIES (Salida: mpg) ===")
print(f"R2 del modelo completo: {modelo_d_mpg.score(X_d.values, y_d.values):.4f}\n")

# Mostrar Betas
betas_d = pd.DataFrame({'Variable': X_d.columns, 'Beta': modelo_d_mpg.coef_})
print("Coeficientes (Betas):")
print(betas_d.to_string(index=False))

# 2. Train-Test Split (40% Entrenamiento)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_d.values, y_d.values, train_size=0.4, random_state=42)

modelo_split_d = LinearRegression()
modelo_split_d.fit(X_train_d, y_train_d)

print("\n=== EVALUACIÓN CON SPLIT 40% ===")
print(f"R2 Entrenamiento: {modelo_split_d.score(X_train_d, y_train_d):.4f}")
print(f"R2 Prueba:        {modelo_split_d.score(X_test_d, y_test_d):.4f}")

=== MODELO CON DUMMIES (Salida: mpg) ===
R2 del modelo completo: 0.8931

Coeficientes (Betas):
Variable      Beta
    disp  0.035546
      hp -0.070507
    drat  1.182830
      wt -4.529776
    qsec  0.367845
      vs  1.930851
      am  1.212116
   cyl_6 -2.648695
   cyl_8 -0.336163
  gear_4  1.114355
  gear_5  2.528396
  carb_2 -0.979354
  carb_3  2.999639
  carb_4  1.091423
  carb_6  4.477569
  carb_8  7.250411

=== EVALUACIÓN CON SPLIT 40% ===
R2 Entrenamiento: 1.0000
R2 Prueba:        -1.3253


### Interpretación de los signos de los Betas con Dummies
Al usar `drop_first=True`, la interpretación cambia. Ahora los Betas de las variables dummy se interpretan en comparación con la categoría que se eliminó (la categoría base).
* **Ejemplo:** La categoría base para cilindros fue `cyl_4`, y el coeficiente para `cyl_6` es negativo, significa que pasar de un motor de 4 cilindros a uno de 6 cilindros reduce el rendimiento (`mpg`) en esa magnitud exacta, asumiendo que el resto de las variables se mantienen constantes.
* Las variables continuas (como el peso `wt` o `hp`) se siguen interpretando igual que en el punto 1.1.

## 2.2 Regresión con variables Dummy (Salida: `qsec`)
Repetimos el procedimiento anterior utilizando el dataset con variables dummy, pero esta vez intentando predecir el tiempo del cuarto de milla (`qsec`).

In [17]:
# === 2.2 REGRESIÓN CON DUMMIES (Salida: qsec) ===

# Usamos el mismo dataframe con dummies, solo cambiamos la 'y'
X_d_qsec = df_dummies.drop('qsec', axis=1)
y_d_qsec = df_dummies['qsec']

# Ajustar modelo completo
modelo_d_qsec = LinearRegression()
modelo_d_qsec.fit(X_d_qsec.values, y_d_qsec.values)

print("=== MODELO CON DUMMIES (Salida: qsec) ===")
print(f"R2 del modelo completo: {modelo_d_qsec.score(X_d_qsec.values, y_d_qsec.values):.4f}\n")

# Mostrar Betas
betas_d_q = pd.DataFrame({'Variable': X_d_qsec.columns, 'Beta': modelo_d_qsec.coef_})
print("Coeficientes (Betas):")
print(betas_d_q.to_string(index=False))

# Train-Test Split (40% Entrenamiento)
X_train_dq, X_test_dq, y_train_dq, y_test_dq = train_test_split(X_d_qsec.values, y_d_qsec.values, train_size=0.4, random_state=42)

modelo_split_dq = LinearRegression()
modelo_split_dq.fit(X_train_dq, y_train_dq)

print("\n=== EVALUACIÓN CON SPLIT 40% ===")
print(f"R2 Entrenamiento: {modelo_split_dq.score(X_train_dq, y_train_dq):.4f}")
print(f"R2 Prueba:        {modelo_split_dq.score(X_test_dq, y_test_dq):.4f}")

=== MODELO CON DUMMIES (Salida: qsec) ===
R2 del modelo completo: 0.9083

Coeficientes (Betas):
Variable      Beta
     mpg  0.027741
    disp  0.003531
      hp -0.002066
    drat  0.107866
      wt  0.810472
      vs  0.265732
      am -1.694294
   cyl_6 -1.104343
   cyl_8 -2.966901
  gear_4  1.332282
  gear_5  0.182317
  carb_2 -0.834413
  carb_3 -0.229304
  carb_4 -1.947034
  carb_6 -1.565241
  carb_8 -1.332317

=== EVALUACIÓN CON SPLIT 40% ===
R2 Entrenamiento: 1.0000
R2 Prueba:        -0.0600


## 3.1 Comparación de los $R^2$ de los ejercicios 1.1 y 2.1 (Salida: `mpg`)

Al comparar ambos ejercicios, observamos que el modelo con variables dummy (2.1) obtiene un mayor $R^2$ en el conjunto de entrenamiento en comparación con el modelo numérico/ordinal (1.1). Sin embargo, su desempeño en el conjunto de prueba es inferior en la mayoría de los escenarios (a excepción de los casos sin regularización o con un $\lambda$ muy bajo).

Este comportamiento se explica por el aumento de la dimensionalidad del dataset. Al convertir variables en dummies, incrementamos la cantidad de predictores. En una muestra tan pequeña (32 observaciones), esto facilita que el modelo memorice el ruido de los datos de entrenamiento (*overfitting* severo), pero pierda drásticamente su capacidad de generalización. La regularización L2 con penalizaciones más altas en el ejercicio 1.1 demostró ser la mejor estrategia para mitigar este problema.


## 3.2 Comparación de los $R^2$ de los ejercicios 1.2 y 2.2 (Salida: `qsec`)

Se observa exactamente la misma tendencia que en el análisis anterior: el modelo que conserva las variables en su formato original (1.2) generaliza mejor y presenta un $R^2$ de prueba superior al modelo con dummies (2.2), particularmente cuando se aplican niveles moderados o altos de regularización.

Esto confirma que, para predecir la aceleración (`qsec`), mantener la naturaleza ordinal de variables como los cilindros (`cyl`) o los cambios (`gear`) resulta en un modelo más robusto. Binarizar estas variables infla artificialmente la cantidad de columnas, empeorando el sobreajuste sin aportar valor predictivo real sobre los datos de prueba.